In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import shutil
import time

src_dir = "/content/drive/MyDrive/HK1-20252026/Steganography/Data/musdb18-hq/train/"
dst_dir = "/content/local_input/"

def copy_ignore_errors(src, dst):
    if not os.path.exists(dst):
        os.makedirs(dst)

    skipped_files = [] # Danh sách lưu tên các file bị lỗi
    success_count = 0

    print(f"--- Bắt đầu copy từ: {src} ---")

    # Duyệt qua cây thư mục
    for root, dirs, files in os.walk(src):
        # Tạo folder đích tương ứng
        rel_path = os.path.relpath(root, src)
        dest_folder = os.path.join(dst, rel_path)
        if not os.path.exists(dest_folder):
            os.makedirs(dest_folder)

        for file in files:
            src_file = os.path.join(root, file)
            dst_file = os.path.join(dest_folder, file)

            # Nếu file đã có rồi thì bỏ qua (Resume)
            if os.path.exists(dst_file):
                continue

            try:
                # Thử copy
                shutil.copy2(src_file, dst_file)
                success_count += 1

                if success_count % 100 == 0:
                    print(f"-> Đã copy {success_count} files...")

            except Exception as e:
                # NẾU LỖI: In ra và BỎ QUA, chạy tiếp vòng lặp
                print(f"[SKIP] Lỗi tại file: {file}")
                # print(f"       Lý do: {e}") # Bỏ comment nếu muốn xem chi tiết lỗi
                skipped_files.append(src_file)

                # Mẹo: Ngủ 1 chút để Drive đỡ bị nghẽn nếu lỗi do quá tải
                time.sleep(0.5)

    print("\n" + "="*30)
    print("KẾT QUẢ:")
    print(f" Thành công: {success_count} files")
    print(f" Bỏ qua (Lỗi): {len(skipped_files)} files")

    if len(skipped_files) > 0:
        print("\nDanh sách 5 file bị lỗi đầu tiên:")
        for f in skipped_files[:5]:
            print(f"- {f}")

copy_ignore_errors(src_dir, dst_dir)

--- Bắt đầu copy từ: /content/drive/MyDrive/HK1-20252026/Steganography/Data/musdb18-hq/train/ ---
[SKIP] Lỗi tại file: Al James - Schoolboy Facination_mixture.wav
[SKIP] Lỗi tại file: Al James - Schoolboy Facination_vocals.wav
[SKIP] Lỗi tại file: Al James - Schoolboy Facination_other.wav
-> Đã copy 100 files...
-> Đã copy 200 files...
-> Đã copy 300 files...
-> Đã copy 400 files...
-> Đã copy 500 files...
-> Đã copy 600 files...
-> Đã copy 700 files...

KẾT QUẢ:
✅ Thành công: 747 files
❌ Bỏ qua (Lỗi): 3 files

Danh sách 5 file bị lỗi đầu tiên:
- /content/drive/MyDrive/HK1-20252026/Steganography/Data/musdb18-hq/train/Al James - Schoolboy Facination_mixture.wav
- /content/drive/MyDrive/HK1-20252026/Steganography/Data/musdb18-hq/train/Al James - Schoolboy Facination_vocals.wav
- /content/drive/MyDrive/HK1-20252026/Steganography/Data/musdb18-hq/train/Al James - Schoolboy Facination_other.wav


In [ ]:

!mkdir -p "/content/local_input"
!mkdir -p "/content/local_output"

print(" Bắt đầu đồng bộ bằng Rsync...")

!rsync -av --info=progress2 "/content/drive/MyDrive/HK1-20252026/Steganography/Data/musdb18-hq/train/" "/content/local_input/"
!cp "/content/drive/MyDrive/HK1-20252026/Steganography/Data/audio-cats-and-dogs/5/cat_24.wav" "/content/local_secret.wav"

import os
local_count = sum([len(files) for r, d, files in os.walk("/content/local_input")])
print(f"\n Đã copy xuống máy ảo: {local_count} file.")

In [ ]:

import numpy as np
import os
PROJECT_PATH = '/content/drive/MyDrive/Git-audio-stego/audio-steganograph/AudioStego/exp'

if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    !pip install -q librosa soundfile tensorflow scikit-learn seaborn
else:
    print(f" Không tìm thấy thư mục tại {PROJECT_PATH}")

!mkdir -p "/content/local_output"



In [ ]:
!python run_case.py \
  --case_id 7 \
  --input_dir "/content/local_input" \
  --secret_file "/content/drive/MyDrive/HK1-20252026/Steganography/Data/audio-cats-and-dogs/5/cat_24.wav" \
  --output_base "/content/local_output"

[*] Case: 7_PhaseCoding | Input: local_input | Limit: full
[*] Log path: logs/benchmark_local_input_7_PhaseCoding_full_20260130_021036.csv
[*] Found: 748 files
[748/748] Zeno - Signs_vocals.wav -> PSNR: 25.84 dB
[DONE] Hoàn tất. Log tại: logs/benchmark_local_input_7_PhaseCoding_full_20260130_021036.csv


In [ ]:
import os
import shutil
import time

SOURCE_DIR = "/content/local_output/7_PhaseCoding"
# Đường dẫn đích trên Drive (giữ nguyên cấu trúc thư mục)
DEST_DIR = "/content/drive/MyDrive/HK1-20252026/Steganography/Ablation/7_PhaseCoding"

print(f" BẮT ĐẦU COPY TỪNG FILE...\nNguồn: {SOURCE_DIR}\nĐích:  {DEST_DIR}")

# Tạo thư mục đích trên Drive
os.makedirs(DEST_DIR, exist_ok=True)

# Lấy danh sách file cần copy
files_to_copy = [f for f in os.listdir(SOURCE_DIR) if f.endswith('.wav') or f.endswith('.csv')]
total_files = len(files_to_copy)
copied_count = 0

start_time = time.time()

for i, filename in enumerate(files_to_copy):
    src_file = os.path.join(SOURCE_DIR, filename)
    dest_file = os.path.join(DEST_DIR, filename)

    # Kiểm tra nếu file đã tồn tại trên Drive với cùng kích thước -> Bỏ qua
    if os.path.exists(dest_file):
        if os.path.getsize(src_file) == os.path.getsize(dest_file):
            print(f"[{i+1}/{total_files}] ⏭ Bỏ qua (Đã có): {filename}")
            copied_count += 1
            continue

    try:
        # Copy file
        shutil.copy2(src_file, dest_file)
        print(f"[{i+1}/{total_files}] ✅ Đã copy: {filename}")
        copied_count += 1

        # Flush bộ nhớ đệm xuống đĩa để đảm bảo file được ghi thật sự
        if (i + 1) % 10 == 0:
            os.sync()

    except OSError as e:
        if e.errno == 28: # Lỗi hết dung lượng
            print(f"\n DỪNG LẠI: Google Drive ĐÃ HẾT DUNG LƯỢNG (Full) tại file thứ {i+1}.")
            print(f"File gây lỗi: {filename}")
            print(f"Kích thước file này: {os.path.getsize(src_file) / (1024*1024):.2f} MB")
            break
        else:
            print(f"\n Lỗi khi copy file {filename}: {e}")

print(f"\n KẾT THÚC QUÁ TRÌNH.")
print(f"Tổng số file đã xử lý: {copied_count}/{total_files}")
print(f"Thời gian chạy: {time.time() - start_time:.2f} giây")

 BẮT ĐẦU COPY TỪNG FILE...
Nguồn: /content/local_output/7_PhaseCoding
Đích:  /content/drive/MyDrive/HK1-20252026/Steganography/Ablation/7_PhaseCoding
[1/692] ✅ Đã copy: Giselle - Moss_vocals.wav
[2/692] ✅ Đã copy: The Easton Ellises - Falcon 69_other.wav
[3/692] ✅ Đã copy: Dark Ride - Burning Bridges_other.wav
[4/692] ✅ Đã copy: Grants - PunchDrunk_vocals.wav
[5/692] ✅ Đã copy: BKS - Too Much_drums.wav
[6/692] ✅ Đã copy: Clara Berry And Wooldog - Stella_other.wav
[7/692] ✅ Đã copy: Signe Jakobsen - What Have You Done To Me_mixture.wav
[8/692] ✅ Đã copy: Spike Mullings - Mike's Sulking_vocals.wav
[9/692] ✅ Đã copy: BKS - Too Much_vocals.wav
[10/692] ✅ Đã copy: Music Delta - Disco_bass.wav
[11/692] ✅ Đã copy: Secretariat - Over The Top_bass.wav
[12/692] ✅ Đã copy: Matthew Entwistle - Dont You Ever_vocals.wav
[13/692] ✅ Đã copy: Tom McKenzie - Directions_drums.wav
[14/692] ✅ Đã copy: Buitraker - Revo X_bass.wav
[15/692] ✅ Đã copy: Ben Carrigan - We'll Talk About It All Tonight_other.wav
[